# Grokking Arithmetic: Replicating the 777-Parameter Transformer

**Goal:** Replicate the grokking phenomenon on modular arithmetic, matching the architecture from [yhavinga/gpt-acc-jax](https://github.com/yhavinga/gpt-acc-jax).

**Reference architecture (777 params):**
- 1 layer, 1 head
- d_model = 7, d_ff = 14 (2× expansion)
- Tied embeddings, no FFN bias
- Learned position embeddings
- LR = 0.02, weight decay = 0.01-1.0

**Our task:** Modular addition (a + b mod 97)

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import random
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Data Generation

Modular arithmetic: `a OP b = c (mod p)`

Using p=97 (prime) so division is always defined via modular inverse.

In [ ]:
class ModularArithmeticDataset(Dataset):
    """
    Dataset for modular arithmetic.
    Format: [a, op, b, =] -> c (all mod p)
    
    Vocabulary:
    - 0 to p-1: number tokens
    - p: '+'
    - p+1: '='
    """
    
    def __init__(self, operation='add', p=97, train=True, train_fraction=0.5, seed=42):
        self.p = p
        self.operation = operation
        
        # Vocabulary: p numbers + op + equals
        self.vocab_size = p + 2
        self.op_token = p
        self.eq_token = p + 1
        
        # Generate all (a, b) pairs
        all_pairs = [(a, b) for a in range(p) for b in range(p)]
        
        # For division, b != 0
        if operation == 'div':
            all_pairs = [(a, b) for a, b in all_pairs if b != 0]
        
        # Split train/test
        random.seed(seed)
        random.shuffle(all_pairs)
        split = int(len(all_pairs) * train_fraction)
        
        self.pairs = all_pairs[:split] if train else all_pairs[split:]
        
    def __len__(self):
        return len(self.pairs)
    
    def compute(self, a, b):
        """Compute a OP b mod p"""
        if self.operation == 'add':
            return (a + b) % self.p
        elif self.operation == 'sub':
            return (a - b) % self.p
        elif self.operation == 'mul':
            return (a * b) % self.p
        elif self.operation == 'div':
            b_inv = pow(b, self.p - 2, self.p)
            return (a * b_inv) % self.p
    
    def __getitem__(self, idx):
        a, b = self.pairs[idx]
        c = self.compute(a, b)
        
        # Input: [a, op, b, =] -> predict c
        input_seq = torch.tensor([a, self.op_token, b, self.eq_token], dtype=torch.long)
        target = torch.tensor(c, dtype=torch.long)
        
        return input_seq, target

# Test
ds = ModularArithmeticDataset('add', p=97, train=True)
print(f"Train size: {len(ds)}")
print(f"Vocab size: {ds.vocab_size}")
x, y = ds[0]
print(f"Sample: {x.tolist()} -> {y.item()}")

## 3. Transformer Architecture (777-Style)

Key optimizations from the 777-param paper:
- Single layer with FFN (d_ff = 2× d_model)
- Tied input/output embeddings
- No bias in FFN
- Learned position embeddings (NOT sinusoidal)

In [ ]:
class CausalSelfAttention(nn.Module):
    """Single-head causal self-attention."""
    
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # Combined QKV projection (more efficient)
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, x):
        B, T, C = x.shape
        
        # Compute Q, K, V
        qkv = self.qkv(x)  # (B, T, 3*C)
        q, k, v = qkv.chunk(3, dim=-1)  # Each (B, T, C)
        
        # Attention scores
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_model)  # (B, T, T)
        
        # Causal mask
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn = attn.masked_fill(mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        
        # Apply attention to values
        out = attn @ v  # (B, T, C)
        return self.proj(out)


class TransformerBlock(nn.Module):
    """Single transformer block: attention + FFN."""
    
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        # FFN with no bias (saves params)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.GELU(),
            nn.Linear(d_ff, d_model, bias=False)
        )
        
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class TinyGrokTransformer(nn.Module):
    """
    Minimal transformer for grokking experiments.
    Matches 777-param paper architecture.
    """
    
    def __init__(self, vocab_size, d_model=7, d_ff=14, n_layers=1, max_len=5):
        super().__init__()
        
        self.d_model = d_model
        self.vocab_size = vocab_size
        
        # Token embeddings
        self.token_embed = nn.Embedding(vocab_size, d_model)
        
        # Learned position embeddings (NOT sinusoidal - critical!)
        self.pos_embed = nn.Embedding(max_len, d_model)
        
        # Transformer layers
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, d_ff) for _ in range(n_layers)
        ])
        
        self.ln_final = nn.LayerNorm(d_model)
        
        # Output head - TIED with input embeddings
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_embed.weight
        
        self._init_weights()
        
    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)
        
    def forward(self, x):
        B, T = x.shape
        
        # Token + position embeddings
        tok_emb = self.token_embed(x)
        pos_emb = self.pos_embed(torch.arange(T, device=x.device))
        x = tok_emb + pos_emb
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_final(x)
        
        # Predict from last position
        logits = self.head(x[:, -1, :])
        return logits


def count_parameters(model):
    """Count trainable parameters (accounting for tied weights)."""
    seen = set()
    total = 0
    for name, param in model.named_parameters():
        if param.data_ptr() not in seen:
            seen.add(param.data_ptr())
            total += param.numel()
            print(f"  {name}: {param.numel():,}")
    return total


# Test model
test_model = TinyGrokTransformer(vocab_size=99, d_model=7, d_ff=14, n_layers=1)
print("\nParameter breakdown:")
total_params = count_parameters(test_model)
print(f"\nTotal: {total_params:,} parameters")
print(f"Target: ~777 (777-param paper used 10-digit addition, we use mod 97)")

## 4. Training with Cosine LR Schedule

Key settings from 777-param paper:
- AdamW with weight decay
- Warmup + cosine decay
- High learning rate (0.02) for tiny models

In [ ]:
def get_lr(step, warmup_steps, total_steps, max_lr, min_lr=1e-5):
    """Cosine learning rate schedule with warmup."""
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


def train_epoch(model, loader, optimizer, device, step, warmup_steps, total_steps, max_lr):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Update learning rate
        lr = get_lr(step, warmup_steps, total_steps, max_lr)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
        step += 1
        
        optimizer.zero_grad()
        logits = model(inputs)
        loss = F.cross_entropy(logits, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == targets).sum().item()
        total += inputs.size(0)
    
    return total_loss / total, correct / total, step


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        logits = model(inputs)
        loss = F.cross_entropy(logits, targets)
        
        total_loss += loss.item() * inputs.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == targets).sum().item()
        total += inputs.size(0)
    
    return total_loss / total, correct / total

In [ ]:
# =============================================================================
# CONFIGURATION - Matched to 777-param paper
# =============================================================================

OPERATION = 'add'  # Start with addition (simplest)
P = 97  # Prime modulus

# Architecture (777-style)
D_MODEL = 7
D_FF = 14  # 2x expansion
N_LAYERS = 1
N_HEADS = 1  # Single head (implicit in our CausalSelfAttention)

# Training
BATCH_SIZE = 512  # Full batch preferred for grokking
NUM_EPOCHS = 100000  # Grokking needs MANY epochs
MAX_LR = 0.02  # High LR for tiny models
WEIGHT_DECAY = 1.0  # High weight decay crucial for grokking
WARMUP_EPOCHS = 50  # ~500 steps with our batch size

# Create datasets
train_ds = ModularArithmeticDataset(OPERATION, p=P, train=True, train_fraction=0.5)
test_ds = ModularArithmeticDataset(OPERATION, p=P, train=False, train_fraction=0.5)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Task: {OPERATION} (mod {P})")
print(f"Train: {len(train_ds):,} | Test: {len(test_ds):,}")
print(f"Steps per epoch: {len(train_loader)}")

# Create model
model = TinyGrokTransformer(
    vocab_size=train_ds.vocab_size,
    d_model=D_MODEL,
    d_ff=D_FF,
    n_layers=N_LAYERS
).to(device)

print(f"\nModel: {count_parameters(model):,} parameters")

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.98)  # Standard transformer betas
)

# Calculate total steps for LR schedule
steps_per_epoch = len(train_loader)
total_steps = NUM_EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch
print(f"Total steps: {total_steps:,} | Warmup: {warmup_steps:,}")

In [ ]:
# =============================================================================
# TRAINING LOOP with Grokking Detection
# =============================================================================

history = {
    'train_loss': [], 'train_acc': [],
    'test_loss': [], 'test_acc': [],
    'lr': []
}

best_test_acc = 0
grokking_epoch = None
global_step = 0

print(f"Training for {NUM_EPOCHS:,} epochs...")
print(f"Watching for grokking (sudden test acc jump from <50% to >99%)")
print("=" * 70)

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc, global_step = train_epoch(
        model, train_loader, optimizer, device,
        global_step, warmup_steps, total_steps, MAX_LR
    )
    test_loss, test_acc = evaluate(model, test_loader, device)
    
    # Get current LR
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    history['lr'].append(current_lr)
    
    # Detect grokking
    if test_acc > best_test_acc:
        if test_acc > 0.99 and best_test_acc < 0.5 and grokking_epoch is None:
            grokking_epoch = epoch
            print(f"\n🎯 GROKKING DETECTED at epoch {epoch}!")
            print(f"   Test accuracy jumped from {best_test_acc*100:.1f}% to {test_acc*100:.1f}%")
        best_test_acc = test_acc
        torch.save(model.state_dict(), 'best_grok_model.pth')
    
    # Log progress (every 2500 epochs or key events)
    if (epoch + 1) % 2500 == 0 or epoch == 0 or (test_acc > 0.99 and best_test_acc < 0.99):
        print(f"Epoch {epoch+1:6d} | "
              f"Train: {train_acc*100:5.1f}% | "
              f"Test: {test_acc*100:5.1f}% | "
              f"LR: {current_lr:.6f}")
    
    # Checkpoint every 10k epochs
    if (epoch + 1) % 10000 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history
        }, f'checkpoint_epoch_{epoch+1}.pth')
    
    # Early stop if grokked
    if test_acc > 0.999:
        print(f"\n🏆 Perfect accuracy at epoch {epoch+1}!")
        break

print("=" * 70)
print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")
print(f"Best Test Accuracy: {best_test_acc*100:.2f}%")
if grokking_epoch:
    print(f"⚡ Grokking occurred at epoch: {grokking_epoch}")

## 5. Visualize Grokking Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy
ax = axes[0, 0]
ax.plot(history['train_acc'], label='Train', alpha=0.7)
ax.plot(history['test_acc'], label='Test', alpha=0.7)
if grokking_epoch:
    ax.axvline(x=grokking_epoch, color='red', linestyle='--', label=f'Grokking @ {grokking_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy over Training')
ax.legend()
ax.grid(True, alpha=0.3)

# Loss
ax = axes[0, 1]
ax.plot(history['train_loss'], label='Train', alpha=0.7)
ax.plot(history['test_loss'], label='Test', alpha=0.7)
if grokking_epoch:
    ax.axvline(x=grokking_epoch, color='red', linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss over Training')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# Learning rate
ax = axes[1, 0]
ax.plot(history['lr'])
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.grid(True, alpha=0.3)

# Train vs Test accuracy (the grokking signature)
ax = axes[1, 1]
ax.scatter(history['train_acc'], history['test_acc'], 
           c=range(len(history['train_acc'])), cmap='viridis', alpha=0.5, s=1)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('Train Accuracy')
ax.set_ylabel('Test Accuracy')
ax.set_title('Train vs Test (color = epoch)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grokking_curves.png', dpi=150)
plt.show()

print("\nSaved: grokking_curves.png")

## 6. Analyze the Grokking Transition

In [ ]:
if grokking_epoch:
    print(f"Grokking Analysis")
    print(f"="*50)
    print(f"Grokking epoch: {grokking_epoch}")
    print(f"")
    print(f"Before grokking (epoch {grokking_epoch - 100}):")
    idx = max(0, grokking_epoch - 100)
    print(f"  Train acc: {history['train_acc'][idx]*100:.1f}%")
    print(f"  Test acc:  {history['test_acc'][idx]*100:.1f}%")
    print(f"")
    print(f"After grokking (epoch {grokking_epoch + 100}):")
    idx = min(len(history['train_acc'])-1, grokking_epoch + 100)
    print(f"  Train acc: {history['train_acc'][idx]*100:.1f}%")
    print(f"  Test acc:  {history['test_acc'][idx]*100:.1f}%")
    print(f"")
    print(f"The model spent {grokking_epoch} epochs memorizing before suddenly generalizing.")
else:
    print("No grokking detected yet.")
    print(f"Current test accuracy: {history['test_acc'][-1]*100:.1f}%")
    print(f"")
    print("Tips if grokking hasn't occurred:")
    print("  1. Train longer (grokking can happen very late)")
    print("  2. Increase weight decay")
    print("  3. Verify train accuracy is near 100% (memorization phase)")